# s09 — Bash output filters

**What this teaches:** how Yoke condenses noisy command output (git status, kubectl logs, npm install, ...) before it reaches the model. YAML rules under `config/filters/` define line-level transforms; the bash tool applies them transparently. The notebook runs the same command twice — filter off then on — so the byte-count delta is visible.

**Concept anchor:** the model's context window is the most precious resource in any agent. Output filters trade a few microseconds of regex work for thousands of tokens you would otherwise burn on noise.

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- An LLM provider configured via env vars (see [s01_loop](../s01_loop/s01_loop.ipynb)).
- The `config/filters/` directory must exist at the repo root — this ships with the project. The notebook walks up the filesystem to find it, so it works no matter where Jupyter is launched from.
- A git working tree (the demo runs `git status`). Any directory inside the Yoke repo qualifies.

## 1. Imports

`fstools` exposes both `RunBash` (the raw tool entry point) and `ConfigureBashOutputFilter`, which toggles the filter pipeline globally. `runtime` + `filepath` are only used to locate `config/filters/`.

In [ ]:
import (
    "context"
    "fmt"
    "os"
    "path/filepath"
    "runtime"

    "github.com/blouargant/yoke/core/agentkit"
    "github.com/blouargant/yoke/core/stream"
    fstools "github.com/blouargant/yoke/core/tools"
)

## 2. Helper

We swap the example's `os.Exit`-based `must` for a panic version so a notebook error doesn't kill the GoNB kernel.

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Locate the filters directory

Filters are repo-local YAML; we walk up from this notebook's directory until we find `config/filters/`. This keeps the example portable — works from `examples/`, from the repo root, or from a sibling worktree.

In [ ]:
locateFiltersDir := func() string {
    _, here, _, _ := runtime.Caller(0)
    dir := filepath.Dir(here)
    for i := 0; i < 6; i++ {
        candidate := filepath.Join(dir, "config", "filters")
        if st, err := os.Stat(candidate); err == nil && st.IsDir() {
            return candidate
        }
        dir = filepath.Dir(dir)
    }
    return "config/filters"
}

filtersDir := locateFiltersDir()
fmt.Println("filters dir:", filtersDir)

## 4. Run the same command twice — raw vs filtered

`ConfigureBashOutputFilter` is process-global, so we flip it off, run the command, then flip it on and run it again. The two outputs are byte-for-byte different — everything the model sees in the filtered run has already been condensed.

In [ ]:
ctx := context.Background()
cmd := "git status"

must(fstools.ConfigureBashOutputFilter(fstools.BashOutputFilterConfig{Enabled: false}))
raw, _ := fstools.RunBash(ctx, fstools.BashIn{Command: cmd})

must(fstools.ConfigureBashOutputFilter(fstools.BashOutputFilterConfig{
    Enabled:    true,
    FiltersDir: filtersDir,
}))
filtered, _ := fstools.RunBash(ctx, fstools.BashIn{Command: cmd})

## 5. Print both outputs and the savings

The savings are the difference in bytes (and the percentage of the raw payload). On a busy repo this can easily be 60–90%.

In [ ]:
fmt.Printf("=== %q raw output: %d bytes ===\n%s\n\n", cmd, len(raw), raw)
fmt.Printf("=== %q filtered output: %d bytes ===\n%s\n\n", cmd, len(filtered), filtered)
denom := len(raw)
if denom < 1 {
    denom = 1
}
fmt.Printf("Filter saved %d bytes (%.0f%%)\n\n", len(raw)-len(filtered), 100*float64(len(raw)-len(filtered))/float64(denom))

## 6. Let the agent run it

With filters still enabled, hand the same command to the agent. The model only sees the condensed view, so its one-sentence summary is grounded in *that*, not in the raw blob.

In [ ]:
llm, err := agentkit.NewModel(ctx)
must(err)
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s09_output_filters",
    Description: "Bash output filter demo.",
    Model:       llm,
    Instruction: "Run the requested bash command and report what it produced.",
    Tools:       fstools.New(),
})
must(err)
r, err := agentkit.Runner("s09", a)
must(err)

prompt := "Run `git status` and tell me in one sentence what state the repo is in."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The byte counts in section 5 should differ — sometimes dramatically, sometimes barely, depending on what the command emitted. Either way, the filtered text is what the model sees in section 6.
- Each YAML file under `config/filters/` matches one command family. Cross-link to [s05_tools](../s05_tools/s05_tools.ipynb) to see the same bash tool without filters; cross-link to [s14_permissions](../s14_permissions/s14_permissions.ipynb) for the orthogonal concern of *what the agent is allowed to run*.

## Try it yourself

1. Change `cmd` to something noisy on your machine (`npm install`, `kubectl get pods -A`, `make -n build`) and rerun sections 4–5 to see how the relevant YAML rule trims it.
2. Add a new YAML file under `config/filters/` matching one of *your* frequently-used commands and watch the savings show up without restarting the kernel.